In [0]:
CREATE OR REPLACE VIEW business_analytics_prd.adhoc_logistica.metricas_vendas_lojas_teste 
  (IdCompra COMMENT 'Id da compra efetuada pelo cliente',
   DtCompra COMMENT 'Data em que o cliente realizou a compra',
   NmAgrupamentoDiretoriaSetor COMMENT 'Dimens]ao com a descrição do setor do produto/sku',
   NmMarca COMMENT 'Dimensão com a descrição da marca do produto/sku',
   NmSku COMMENT 'Dimensão com a descrição do produto/sku',
   TotalPedidos COMMENT 'Medida com Total de pedidos únicos')
  WITH METRICS
  LANGUAGE YAML
  COMMENT 'Metric view de vendas colocadas e aprovadas no site/e-commerce/online/on contendo valores e quantidade de itens/peças, sendo relatório contendo dia e hora da venda feita'
  AS $$
  version: 0.1

  # --- Example definition with filter, dimensions and measures ---
  #
  source: data_engineering_prd.app_venda.vendaaprovadalojafisicaflash

  #
  joins: 
    - source: data_engineering_prd.app_venda.mercadoria
      name: mercadoria
      on: SkuLojaReu = CdSkuLojaReu and StUltimaVersaoMercadoria = 'Y'

  filter: to_date(DtDocumento, 'yyyyMMdd') >= CURRENT_DATE

  dimensions:
    # Id do pedido da compra efetuada pelo cliente
    - name: IdCompra
      expr: CdDocumento
    # Data em que o cliente realizou a compra
    - name: DtCompra
      expr: to_date(DtDocumento, 'yyyyMMdd')

    # Nome descritivo da diretorio a qual a compra pertence
    - name: NmAgrupamentoDiretoriaSetor
      expr: trim(mercadoria.NmAgrupamentoDiretoriaSetor)
    
    # Descrição da marca ao qual o produto vendido pertence
    - name: NmMarca
      expr: trim(mercadoria.NmMarca)

    # Descrição do produto vendido
    - name: NmSku
      expr: trim(mercadoria.NmSku)

  measures:
    # Total de pedidos únicos vendidos
    - name: TotalPedidos
      expr: count(distinct IdCompra)

  #   - name: filtered 
  #     expr: count(*) filter (where column_name = 'some value')
  $$;
 

In [0]:
%sql
CREATE OR REPLACE VIEW business_analytics_prd.adhoc_logistica.metricas_vendas_site_valor
  (IdCompra COMMENT 'Número da compra/pedido que o cliente gerou na companhia',
   DtCompra COMMENT 'Data em que a venda foi realizada',
   NmTipoNegocio COMMENT 'Campo que identifica o tipo de negocio referente a venda realizada podendo ser: "3P" e "3P_HO" (ambos pertencem as vendas em canais de marketplace ou simplesmente 3P, o que tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2C" e "B2C_HO" (ambos pertencem as vendas em canais de clientes pessoa física simplesmente vendas do 1P, o quem tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2B" (vendas em canais de clientes B2B)',
   NmBandeira COMMENT 'Bandeira da companhia que a venda foi realizada podendo ser: "PONTOFRIO", "EXTRA" e "CASASBAHIA" ',
   CdSkuSite COMMENT 'Código do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
   CdSkuLoja COMMENT 'Código do SKU/Produto na loja da venda realizada ao cliente',
   NmAgrupamentoDiretoriaSetor COMMENT 'Dimensão com a descrição da diretoria do produto/sku',
   NmSetorGerencial COMMENT 'Setor/Departamento no site/e-commerce/online ao qual a venda pertence',
   NmMarca COMMENT 'Marca do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
   NmSku COMMENT 'Dimensão com a descrição do produto/sku',
   DsCanalMarketing COMMENT 'Canal de marketing onde a venda foi estimulada.',
   VrProdutoRateado COMMENT 'Valor de venda do produto referente a forma de pagamento utilizada. Não inclui os valores de juros e frete.',
   RealGMV COMMENT 'Terminologia do painel funil de vendas, relacionada a vendas captadas sem juros e com frete. É a soma do VrProdutoRateado com VrFreteRateado.',
   RealGMVcomJuros COMMENT 'Terminologia do painel funil de vendas, relacionada a vendas captadas com juros e frete. É a soma do VrProdutoRateado com VrFreteRateado.',
   RealOrders COMMENT 'Terminologia do painel funil de vendas, relacionada a vendas captadas. É a soma do VrProdutoRateado. Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).',
   RealAprovado COMMENT 'Terminologia do painel funil de vendas, pode ser chamado de "venda aprovadas" ou "real app". É a soma do VrProdutoRateado quando o pedido é aprovado (IdFlagAprovado=true). Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).',
   VrMargem COMMENT 'Margem da venda. É o calculo do VrProdutoRateado retirando os custos de impostos.',
   VrReceitaLiquidaSemImpostos COMMENT 'Receita liquida retirando os impostos.',
   VrDesconto COMMENT 'Desconto realizado na venda.')
  WITH METRICS
  LANGUAGE YAML
  COMMENT 'Metricas baseasdas na view de vendas colocadas e aprovadas do site/e-commerce/online/on contendo valores , sendo relatório contendo dia e hora da venda feita'
  AS $$
  version: 0.1

  # --- Example definition with filter, dimensions and measures ---
  #
  source: data_engineering_prd.app_venda.vendacolocadaaprovadasiterateado

  #
  joins: 
    - source: data_engineering_prd.app_venda.mercadoria
      name: merc
      on: merc.CdSkuSite = source.CdSkuSite and merc.StUltimaVersaoMercadoria = 'Y' and not(upper(TRIM(merc.NmSetorGerencial)) in ("BORAVAREJAR", "CARTAO PRESENTE"))
    
    - source: data_engineering_prd.app_venda.canalmarketing 
      name: cm 
      on: source.NmParceiro = cm.DsUtmSource

  filter: source.IdLojista NOT IN (select IdLojista from data_engineering_prd.app_marketplace.cadastrolojista where NmClusterLojista in ('VENDA_DAT', 'Extra/Ztoy', 'Via Varejo', 'DEPÓSITO VV')) and merc.CdSkuSite is not null

  dimensions:
    # Id do pedido da compra efetuada pelo cliente
    - name: IdCompra
      expr: source.IdCompra

    # Data em que o cliente realizou a compra
    - name: DtCompra
      expr: to_date(source.DtCompra, 'yyyyMMdd')

    - name: NmTipoNegocio
      expr: source.NmTipoNegocio

    - name: NmBandeira
      expr: source.NmBandeira

    # Código do SKU/Produto no site/e-commerce/online da venda realizada ao cliente
    - name: CdSkuSite
      expr: source.CdSkuSite
    
    # Código do SKU/Produto na loja da venda realizada ao cliente
    - name: CdSkuLoja
      expr: merc.CdSkuLoja

    # Nome descritivo da diretorio a qual a compra pertence
    - name: NmAgrupamentoDiretoriaSetor
      expr: trim(merc.NmAgrupamentoDiretoriaSetor)
    
    # Setor/Departamento no site/e-commerce/online ao qual a venda pertence
    - name: NmSetorGerencial
      expr: trim(merc.NmSetorGerencial)
    
    # Descrição da marca ao qual o produto vendido pertence
    - name: NmMarca
      expr: trim(merc.NmMarca)

    # Descrição do produto vendido
    - name: NmSku
      expr: trim(merc.NmSku)

    # Canal de marketing onde a venda foi estimulada.
    - name: DsCanalMarketing
      expr: trim(cm.NmCanalMarketing)

  measures:
    # Valor de venda do produto referente a forma de pagamento utilizada. Não inclui os valores de juros e frete.
    - name: VrProdutoRateado
      expr: sum(source.VrProdutoRateado)

    # Terminologia do painel funil de vendas, relacionada a vendas captadas sem juros e com frete. É a soma do VrProdutoRateado com VrFreteRateado.
    - name: RealGMV
      expr: sum(coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0))
    
    # Terminologia do painel funil de vendas, relacionada a vendas captadas com juros e frete. É a soma do VrProdutoRateado com VrFreteRateado.
    - name: RealGMVcomJuros
      expr: sum(coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0) + coalesce(source.VrJurosRateado,0))
    
    # Terminologia do painel funil de vendas, relacionada a vendas captadas. É a soma do VrProdutoRateado. Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).
    - name: RealOrders
      expr: sum(case when source.NmTipoNegocio in ('3P','3P_HO') then coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0) else coalesce(source.VrProdutoRateado,0) end)

    # Terminologia do painel funil de vendas, pode ser chamado de "venda aprovadas" ou "real app". É a soma do VrProdutoRateado quando o pedido é aprovado (IdFlagAprovado=true). Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).
    - name: RealAprovado
      expr: sum(case when source.IdFlagAprovado=TRUE and source.NmTipoNegocio in ('3P','3P_HO') then coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0)
        when source.IdFlagAprovado=TRUE then source.VrProdutoRateado else 0 end)

    # Margem da venda. É o calculo do VrProdutoRateado retirando os custos de impostos.
    - name: VrMargem
      expr: sum(case when merc.CdSetorGerencial<>20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrIcmsVenda +source.VrCustoContabilFilialSkuMarkup) + (source.VrRessarcimentoCreditoPresumido+source.VrRessarcimentoQuebraCadeia+source.VrRessarcimentoSTF) when merc.CdSetorGerencial=20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrCustoContabilFilialSkuMarkup) + (source.VrRessarcimentoCreditoPresumido+source.VrRessarcimentoQuebraCadeia+source.VrRessarcimentoSTF) else 0 end)
    
    # Receita liquida retirando os impostos.
    - name: VrReceitaLiquidaSemImpostos
      expr: sum(case when merc.CdSetorGerencial<>20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrIcmsVenda) when merc.CdSetorGerencial=20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS) else 0 end)
    
    # Desconto realizado na venda.
    - name: VrDesconto
      expr: sum(source.VrProdutoSemDesconto - source.VrProdutoRateado)

  $$;
 

In [0]:
# Obter colunas da tabela vendaaprovadalojasfisicas
df1 = spark.table("data_engineering_prd.app_venda.vendacolocadaaprovadasiterateado")
columns_df1 = set(df1.columns)
 
# Obter colunas da tabela vendaaprovadalojafisicaflash
df2 = spark.table("data_engineering_prd.app_venda.vendaaprovadalojafisicaflash")
columns_df2 = set(df2.columns)
 
# Encontrar colunas comuns
common_columns = columns_df1.intersection(columns_df2)
display(common_columns)
 

In [0]:
%sql
CREATE OR REPLACE VIEW business_analytics_prd.adhoc_logistica.metricas_vendas_site_valor_qtd
  (
  CdSkuSite COMMENT 'Código do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  CdSkuLoja COMMENT 'Código do SKU/Produto na loja da venda realizada ao cliente',
  NmSku COMMENT 'Descrição/Nome do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  NmMarca COMMENT 'Marca do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  DtCompra COMMENT 'Data em que a venda foi realizada, em formato de numero inteiro "yyyyMMdd". Sempre transforme em data na visualização final.',
  HrVenda COMMENT 'Hora do dia em que a venda foi realizada',
  NmAgrupamentoDiretoriaSetor COMMENT 'Diretoria ao qual a mercadoria pertence',
  NmSetorGerencial COMMENT 'Setor/Departamento no site/e-commerce/online ao qual a venda pertence',
  CdSetorGerencial COMMENT 'Código do setor/Departamento no site/e-commerce/online ao qual a venda pertence',
  NmTipoNegocio COMMENT 'Campo que identifica o tipo de negocio referente a venda realizada podendo ser: "3P" e "3P_HO" (ambos pertencem as vendas em canais de marketplace ou simplesmente 3P, o que tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2C" e "B2C_HO" (ambos pertencem as vendas em canais de clientes pessoa física simplesmente vendas do 1P, o quem tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2B" (vendas em canais de clientes B2B)',
  IdCanalVenda COMMENT 'Canal através do qual a compra/pedido foi realizada podendo ser "SITE" (Através do navegador utilizando um desktop/PC), "MOBI" (Através do navegador utilizando um smartphone pode ser entendido também como "MSITE"), "APP" (Através do aplicativo Android ou IOS que existem na galeria de Aplicativos do smartphone), "TVEN" (Televendas - Vendas por telefone),"STLC", "B2B" (canais de vendas de parceiros da companhia), "TMCM","CHTN","QOSK","TVLC","CTGO","BYER" ',
  DsCanalVenda COMMENT 'Descrição dos Canais de vendas utilizados para gerar a venda',
  NmBandeira COMMENT 'Bandeira da companhia que a venda foi realizada podendo ser: "PONTOFRIO", "EXTRA" e "CASASBAHIA" ',
  IdFlagAprovado COMMENT 'Identifica se a venda foi aprovada ou nao podendo ser: false ou true.',
  DtAprovacao COMMENT 'Data em que ocorreu a aprovação da compra/pedido, em formato de numero inteiro "yyyyMMdd". Sempre transforme em data na visualização final. ',
  IdFilial COMMENT '',
  IdFilialLoja COMMENT '',
  NmUfOrigem COMMENT 'Estado da federeção do Brasil em que se originou a compra/pedido',
  NmUfEntrega COMMENT 'Estado da federeção do Brasil em que o pedido foi ou será entregue ao cliente',
  NmTipoPagamento COMMENT 'Tipo do pagamento realizado pelo cliente',
  QtParcela COMMENT 'Quantidade de parcelas em que o cliente optou por realizar o pagamento, sempre está associado a forma de pagamento',
  NmTipoCelula COMMENT '',
  NmTipoEntregaSite COMMENT 'Tipo ou meio de entrega a ser realizado para o cliente ',
  DsCanalMarketing COMMENT 'Canal de marketing onde a venda foi estimulada.',
  VrProdutoRateado COMMENT 'Valor de venda do produto referente a forma de pagamento utilizada. Não inclui os valores de juros e frete',
  VrFreteRateado COMMENT 'Valor do frete cobrado ao cliente para realizar a entrega da venda',
  VrJurosRateado COMMENT 'Valor do juros cobrado ao cliente quando efetuado uma venda a prazo',
  RealGMV COMMENT 'Terminologia do painel funil de vendas. É a soma do VrProdutoRateado com VrFreteRateado. Não inclui os valores de juros',
  RealGMVcomJuros COMMENT 'Terminologia do painel funil de vendas. É a soma do VrProdutoRateado com VrFreteRateado e VrJurosRateado',
  RealOrders COMMENT 'Terminologia do painel funil de vendas. É a soma do VrProdutoRateado. Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado)',
  RealAprovado COMMENT 'Terminologia do painel funil de vendas, pode ser chamado de "venda aprovadas" ou "real app". É a soma do VrProdutoRateado quando o pedido é aprovado (IdFlagAprovado=true). Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado)',
  VrMargem COMMENT 'Margem da venda. É o calculo do VrProdutoRateado retirando os custos de impostos',
  VrReceitaLiquidaSemImpostos COMMENT 'Receita liquida retirando os impostos',
  VrDesconto COMMENT 'Desconto realizado na venda.',
  QtItem COMMENT 'Quantidade de itens vendidos do SKU/Produto/Mercadoria no site/e-commerce/online da venda realizada ao cliente',
  VrFreteRateadoAprovado COMMENT 'Valor do frete cobrado ao cliente para realizar a entrega da venda, somente de vendas aprovadas',
  VrJurosRateadoAprovado COMMENT 'Valor do juros cobrado ao cliente quando efetuado uma venda a prazo, somente de vendas aprovadas'
  )
  WITH METRICS
  LANGUAGE YAML
  COMMENT 'Metricas baseasdas na view de vendas colocadas e aprovadas do site/e-commerce/online/on contendo valores , sendo relatório contendo dia e hora da venda feita'
  AS $$
  version: 0.1

  # --- Example definition with filter, dimensions and measures ---
  #
  source: select CdSkuSite,IdLojista,NmTipoNegocio,NmBandeira,Idcompra,IdCompraEntrega,DtCompra,IdHora,IdCanalVenda,IdFlagAprovado,DtAprovacao,IdFilial,IdFilialLoja,NmUfOrigem,NmUfEntrega,IdTipoEntregaSite,IdMeioPagamento,IdTipoCelula,NmParceiro,0 as QtItem,VrProdutoRateado,VrFreteRateado,VrJurosRateado,VrCofins,VrPIS,VrIcmsVenda,VrCustoContabilFilialSkuMarkup,VrRessarcimentoCreditoPresumido,VrRessarcimentoQuebraCadeia,VrRessarcimentoSTF,VrProdutoSemDesconto from data_engineering_prd.app_venda.vendacolocadaaprovadasiterateado union all select CdSkuSite,IdLojista,NmTipoNegocio,NmBandeira,Idcompra,IdCompraEntrega,DtCompra,IdHora,IdCanalVenda,IdFlagAprovado,DtAprovacao,IdFilial,IdFilialLoja,NmUfOrigem,NmUfEntrega,IdTipoEntregaSite,null as IdMeioPagamento,IdTipoCelula,NmParceiro,QtItem,0 as VrProdutoRateado,0 as VrFreteRateado,0 as VrJurosRateado,0 as VrCofins,0 as VrPIS,0 as VrIcmsVenda,0 as VrCustoContabilFilialSkuMarkup,0 as VrRessarcimentoCreditoPresumido,0 as VrRessarcimentoQuebraCadeia,0 as VrRessarcimentoSTF,0 as VrProdutoSemDesconto from data_engineering_prd.app_venda.vendacolocadaaprovadasitenaorateado

  #
  joins:
    - source: data_engineering_prd.app_venda.mercadoria
      name: merc
      on: merc.CdSkuSite = source.CdSkuSite and merc.StUltimaVersaoMercadoria = 'Y' and not(upper(TRIM(merc.NmSetorGerencial)) in ("BORAVAREJAR", "CARTAO PRESENTE"))

    - source: data_engineering_prd.app_venda.canalmarketing 
      name: cm 
      on: source.NmParceiro = cm.DsUtmSource
    
    - source: data_engineering_prd.app_venda.tipoentregasite 
      name: tp_ent
      on: tp_ent.IdTipoEntregaSite = source.IdTipoEntregaSite

    - source: data_engineering_prd.app_venda.meiopagamento 
      name: fma_pgto
      on: fma_pgto.IdMeioPagamento = source.IdMeioPagamento
    
    - source: data_engineering_prd.app_venda.tipocelula 
      name: tp_cel
      on: tp_cel.IdTipoCelula = source.IdTipoCelula
    
    - source: data_engineering_prd.context_site.canalvenda 
      name: canal
      on: source.IdCanalVenda = canal.idcanalvenda and source.NmBandeira = canal.bandeira
    
  filter: source.IdLojista NOT IN (select IdLojista from data_engineering_prd.app_marketplace.cadastrolojista where NmClusterLojista in ('VENDA_DAT', 'Extra/Ztoy', 'Via Varejo', 'DEPÓSITO VV')) and source.CdSkuSite is not null and tp_ent.IdTipoEntregaSite is not null

  dimensions:
    - name: CdSkuSite
      expr: source.CdSkuSite

    - name: CdSkuLoja
      expr: merc.CdSkuLoja
    
    - name: NmSku
      expr: merc.NmSku

    - name: NmMarca
      expr: merc.NmMarca

    - name: DtCompra
      expr: source.DtCompra

    - name: HrVenda
      expr: source.IdHora

    - name: NmAgrupamentoDiretoriaSetor
      expr: trim(merc.NmAgrupamentoDiretoriaSetor)

    - name: NmSetorGerencial
      expr: trim(merc.NmSetorGerencial)
    
    - name: CdSetorGerencial
      expr: merc.CdSetorGerencial

    - name: NmTipoNegocio
      expr: source.NmTipoNegocio

    - name: IdCanalVenda
      expr: source.IdCanalVenda

    - name: DsCanalVenda
      expr: canal.nome

    - name: NmBandeira
      expr: source.NmBandeira

    - name: IdFlagAprovado
      expr: source.IdFlagAprovado

    - name: DtAprovacao
      expr: source.DtAprovacao

    - name: IdFilial
      expr: source.IdFilial

    - name: IdFilialLoja
      expr: source.IdFilialLoja

    - name: NmUfOrigem
      expr: source.NmUfOrigem

    - name: NmUfEntrega
      expr: source.NmUfEntrega

    - name: NmTipoPagamento
      expr: fma_pgto.NmTipoPagamento

    - name: QtParcela
      expr: fma_pgto.QtParcela

    - name: NmTipoCelula
      expr: tp_cel.NmTipoCelula

    - name: NmTipoEntregaSite
      expr: tp_ent.NmTipoEntregaSite

    - name: DsCanalMarketing
      expr: cm.NmCanalMarketing

  measures:
    # Valor de venda do produto referente a forma de pagamento utilizada. Não inclui os valores de juros e frete.
    - name: VrProdutoRateado
      expr: sum(source.VrProdutoRateado)
    
    - name: VrFreteRateado
      expr: sum(source.VrFreteRateado)
    
    - name: VrJurosRateado
      expr: sum(source.VrJurosRateado)

    # Terminologia do painel funil de vendas, relacionada a vendas captadas sem juros e com frete. É a soma do VrProdutoRateado com VrFreteRateado.
    - name: RealGMV
      expr: sum(coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0))
    
    # Terminologia do painel funil de vendas, relacionada a vendas captadas com juros e frete. É a soma do VrProdutoRateado com VrFreteRateado.
    - name: RealGMVcomJuros
      expr: sum(coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0) + coalesce(source.VrJurosRateado,0))
    
    # Terminologia do painel funil de vendas, relacionada a vendas captadas. É a soma do VrProdutoRateado. Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).
    - name: RealOrders
      expr: sum(case when source.NmTipoNegocio in ('3P','3P_HO') then coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0) else coalesce(source.VrProdutoRateado,0) end)

    # Terminologia do painel funil de vendas, pode ser chamado de "venda aprovadas" ou "real app". É a soma do VrProdutoRateado quando o pedido é aprovado (IdFlagAprovado=true). Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).
    - name: RealAprovado
      expr: sum(case when source.IdFlagAprovado=TRUE and source.NmTipoNegocio in ('3P','3P_HO') then coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0)
        when source.IdFlagAprovado=TRUE then source.VrProdutoRateado else 0 end)

    # Margem da venda. É o calculo do VrProdutoRateado retirando os custos de impostos.
    - name: VrMargem
      expr: sum(case when merc.CdSetorGerencial<>20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrIcmsVenda +source.VrCustoContabilFilialSkuMarkup) + (source.VrRessarcimentoCreditoPresumido+source.VrRessarcimentoQuebraCadeia+source.VrRessarcimentoSTF) when merc.CdSetorGerencial=20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrCustoContabilFilialSkuMarkup) + (source.VrRessarcimentoCreditoPresumido+source.VrRessarcimentoQuebraCadeia+source.VrRessarcimentoSTF) else 0 end)
    
    # Receita liquida retirando os impostos.
    - name: VrReceitaLiquidaSemImpostos
      expr: sum(case when merc.CdSetorGerencial<>20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrIcmsVenda) when merc.CdSetorGerencial=20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS) else 0 end)
    
    # Desconto realizado na venda.
    - name: VrDesconto
      expr: sum(source.VrProdutoSemDesconto - source.VrProdutoRateado)

    - name: QtItem
      expr: sum(source.QtItem)
    
    - name: VrFreteRateadoAprovado
      expr: sum(case when source.IdFlagAprovado=TRUE then source.VrFreteRateado else 0 end)
    
    - name: VrJurosRateadoAprovado
      expr: sum(case when source.IdFlagAprovado=TRUE then source.VrJurosRateado else 0 end)
  $$;
 

In [0]:
%sql
CREATE OR REPLACE VIEW business_analytics_prd.adhoc_logistica.metricas_vendas_site_qtd
  (  IdCompra
  COMMENT 'Número da compra/pedido que o cliente gerou na companhia',
  IdCompraEntrega
  COMMENT 'Número da entrega/documento relacionado a compra/pedido que o cliente gerou na companhia',
  CdSkuSite
  COMMENT 'Código do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  CdSkuLoja
  COMMENT 'Código do SKU/Produto na loja da venda realizada ao cliente',
  NmSku
  COMMENT 'Descrição/Nome do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  NmMarca
  COMMENT 'Marca do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  DtCompra
  COMMENT 'Data em que a venda foi realizada, em formato de numero inteiro "yyyyMMdd".',
  HrVenda
  COMMENT 'Hora do dia em que a venda foi realizada',
  NmSetorGerencial
  COMMENT 'Setor/Departamento no site/e-commerce/online ao qual a venda pertence',
  NmAgrupamentoDiretoriaSetor
  COMMENT 'Diretoria ao qual a mercadoria pertence',
  NmTipoNegocio
  COMMENT 'Campo que identifica o tipo de negocio referente a venda realizada podendo ser: "3P" e "3P_HO" (ambos pertencem as vendas em canais de marketplace ou simplesmente 3P, o que tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2C" e "B2C_HO" (ambos pertencem as vendas em canais de clientes pessoa física simplesmente vendas do 1P, o quem tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2B" (vendas em canais de clientes B2B)',
  IdCanalVenda
  COMMENT 'Canal através do qual a compra/pedido foi realizada podendo ser "SITE" (Através do navegador utilizando um desktop/PC), "MOBI" (Através do navegador utilizando um smartphone pode ser entendido também como "MSITE"), "APP" (Através do aplicativo Android ou IOS que existem na galeria de Aplicativos do smartphone), "TVEN" (Televendas - Vendas por telefone),"STLC", "B2B" (canais de vendas de parceiros da companhia), "TMCM","CHTN","QOSK","TVLC","CTGO","BYER" ',
  DsCanalVenda
  COMMENT 'Descrição dos Canais de vendas utilizados para gerar a venda',
  NmBandeira
  COMMENT 'Bandeira da companhia que a venda foi realizada podendo ser: "PONTOFRIO", "EXTRA" e "CASASBAHIA" ',
  IdFlagAprovado
  COMMENT 'Identifica se a venda foi aprovada ou nao podendo ser: false ou true.',
  DtAprovacao
  COMMENT 'Data em que ocorreu a aprovação da compra/pedido',
  IdFilial
  COMMENT '',
  IdFilialLoja
  COMMENT '',
  NmUfOrigem
  COMMENT 'Estado da federeção do Brasil em que se originou a compra/pedido',
  NmUfEntrega
  COMMENT 'Estado da federeção do Brasil em que o pedido foi ou será entregue ao cliente',
  NmTipoCelula
  COMMENT '',
  NmTipoEntregaSite
  COMMENT 'Tipo ou meio de entrega a ser realizado para o cliente ',
  DsCanalMarketing
  COMMENT 'Canal de marketing onde a venda foi estimulada.',
  QtItem
  COMMENT 'Quantidade de itens vendidos do SKU/Produto/Mercadoria no site/e-commerce/online da venda realizada ao cliente',
  QtPedidos
  COMMENT 'Quantidade de pedidos captados ou aprovados no site/e-commerce/online da venda realizada ao cliente')
  WITH METRICS
  LANGUAGE YAML
  COMMENT 'Metricas baseadas no cubo de vendas, com informações de vendas colocadas e aprovadas do site/e-commerce/online/on contendo a quantidade de itens vendidos, sendo relatório contendo dia e hora da venda feita'
  AS $$
  version: 0.1

  # --- Example definition with filter, dimensions and measures ---
  #
  source: data_engineering_prd.app_venda.vendacolocadaaprovadasitenaorateado

  #
  joins: 
    - source: data_engineering_prd.app_venda.mercadoria
      name: merc
      on: merc.CdSkuSite = source.CdSkuSite and merc.StUltimaVersaoMercadoria = 'Y' and not(upper(TRIM(merc.NmSetorGerencial)) in ("BORAVAREJAR", "CARTAO PRESENTE"))
    
    - source: data_engineering_prd.app_venda.canalmarketing 
      name: cm 
      on: source.NmParceiro = cm.DsUtmSource
    
    - source: data_engineering_prd.app_venda.tipoentregasite 
      name: tp_ent
      on: tp_ent.IdTipoEntregaSite = source.IdTipoEntregaSite
    
    - source: data_engineering_prd.app_venda.tipocelula 
      name: tp_cel
      on: tp_cel.IdTipoCelula = source.IdTipoCelula
    
    - source: data_engineering_prd.context_site.canalvenda 
      name: canal
      on: source.IdCanalVenda = canal.idcanalvenda and source.NmBandeira = canal.bandeira

  filter: source.IdLojista NOT IN (select IdLojista from data_engineering_prd.app_marketplace.cadastrolojista where NmClusterLojista in ('VENDA_DAT', 'Extra/Ztoy', 'Via Varejo', 'DEPÓSITO VV')) and merc.CdSkuSite is not null and tp_ent.IdTipoEntregaSite is not null

  dimensions:
    - name: IdCompra
      expr: source.IdCompra

    - name: IdCompraEntrega
      expr: source.IdCompraEntrega

    - name: CdSkuSite
      expr: merc.CdSkuSite

    - name: CdSkuLoja
      expr: merc.CdSkuLoja
    
    - name: NmSku
      expr: merc.NmSku

    - name: NmMarca
      expr: merc.NmMarca

    - name: DtCompra
      expr: source.DtCompra

    - name: HrVenda
      expr: source.IdHora

    - name: NmSetorGerencial
      expr: trim(merc.NmSetorGerencial)

    - name: NmAgrupamentoDiretoriaSetor
      expr: trim(merc.NmAgrupamentoDiretoriaSetor)

    - name: NmTipoNegocio
      expr: source.NmTipoNegocio

    - name: IdCanalVenda
      expr: source.IdCanalVenda

    - name: DsCanalVenda
      expr: canal.nome

    - name: NmBandeira
      expr: source.NmBandeira

    - name: IdFlagAprovado
      expr: source.IdFlagAprovado

    - name: DtAprovacao
      expr: to_date(source.DtAprovacao, 'yyyyMMdd')

    - name: IdFilial
      expr: source.IdFilial

    - name: IdFilialLoja
      expr: source.IdFilialLoja

    - name: NmUfOrigem
      expr: source.NmUfOrigem

    - name: NmUfEntrega
      expr: source.NmUfEntrega

    - name: NmTipoCelula
      expr: tp_cel.NmTipoCelula

    - name: NmTipoEntregaSite
      expr: tp_ent.NmTipoEntregaSite

    - name: DsCanalMarketing
      expr: cm.NmCanalMarketing

  measures:
    - name: QtItem
      expr: sum(source.QtItem)
    
    - name: QtPedidos
      expr: count(distinct source.IdCompra)

  $$;
 

In [0]:
%sql
CREATE OR REPLACE VIEW business_analytics_prd.adhoc_logistica.metricas_vendas_site_valor
  (  IdCompra
  COMMENT 'Número da compra/pedido que o cliente gerou na companhia',
  IdCompraEntrega
  COMMENT 'Número da entrega/documento relacionado a compra/pedido que o cliente gerou na companhia',
  CdSkuSite
  COMMENT 'Código do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  CdSkuLoja
  COMMENT 'Código do SKU/Produto na loja da venda realizada ao cliente',
  NmSku
  COMMENT 'Descrição/Nome do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  NmMarca
  COMMENT 'Marca do SKU/Produto no site/e-commerce/online da venda realizada ao cliente',
  DtCompra
  COMMENT 'Data em que a venda foi realizada, em formato de numero inteiro "yyyyMMdd".',
  HrVenda
  COMMENT 'Hora do dia em que a venda foi realizada',
  NmSetorGerencial
  COMMENT 'Setor/Departamento no site/e-commerce/online ao qual a venda pertence',
  NmAgrupamentoDiretoriaSetor
  COMMENT 'Diretoria ao qual a mercadoria pertence',
  NmTipoNegocio
  COMMENT 'Campo que identifica o tipo de negocio referente a venda realizada podendo ser: "3P" e "3P_HO" (ambos pertencem as vendas em canais de marketplace ou simplesmente 3P, o que tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2C" e "B2C_HO" (ambos pertencem as vendas em canais de clientes pessoa física simplesmente vendas do 1P, o quem tem o sufixo "_HO" refere-se as vendas através do vendedor online ou simplesmente VO), "B2B" (vendas em canais de clientes B2B)',
  IdCanalVenda
  COMMENT 'Canal através do qual a compra/pedido foi realizada podendo ser "SITE" (Através do navegador utilizando um desktop/PC), "MOBI" (Através do navegador utilizando um smartphone pode ser entendido também como "MSITE"), "APP" (Através do aplicativo Android ou IOS que existem na galeria de Aplicativos do smartphone), "TVEN" (Televendas - Vendas por telefone),"STLC", "B2B" (canais de vendas de parceiros da companhia), "TMCM","CHTN","QOSK","TVLC","CTGO","BYER" ',
  DsCanalVenda
  COMMENT 'Descrição dos Canais de vendas utilizados para gerar a venda',
  NmBandeira
  COMMENT 'Bandeira da companhia que a venda foi realizada podendo ser: "PONTOFRIO", "EXTRA" e "CASASBAHIA" ',
  IdFlagAprovado
  COMMENT 'Identifica se a venda foi aprovada ou nao podendo ser: false ou true.',
  DtAprovacao
  COMMENT 'Data em que ocorreu a aprovação da compra/pedido',
  IdFilial
  COMMENT '',
  IdFilialLoja
  COMMENT '',
  NmUfOrigem
  COMMENT 'Estado da federeção do Brasil em que se originou a compra/pedido',
  NmUfEntrega
  COMMENT 'Estado da federeção do Brasil em que o pedido foi ou será entregue ao cliente',
  NmTipoPagamento
  COMMENT 'Tipo do pagamento realizado pelo cliente',
  QtParcela
  COMMENT 'Quantidade de parcelas em que o cliente optou por realizar o pagamento, sempre está associado a forma de pagamento',
  NmTipoCelula
  COMMENT '',
  NmTipoEntregaSite
  COMMENT 'Tipo ou meio de entrega a ser realizado para o cliente ',
  DsCanalMarketing
  COMMENT 'Canal de marketing onde a venda foi estimulada.',
  VrProdutoRateado
  COMMENT 'Valor de venda do produto referente a forma de pagamento utilizada. Não inclui os valores de juros e frete',
  VrFreteRateado
  COMMENT 'Valor do frete cobrado ao cliente para realizar a entrega da venda',
  VrJurosRateado
  COMMENT 'Valor do juros cobrado ao cliente quando efetuado uma venda a prazo',
  RealGMV
  COMMENT 'Terminologia do painel funil de vendas. É a soma do VrProdutoRateado com VrFreteRateado. Não inclui os valores de juros',
  RealGMVcomJuros
  COMMENT 'Terminologia do painel funil de vendas. É a soma do VrProdutoRateado com VrFreteRateado e VrJurosRateado',
  RealOrders
  COMMENT 'Terminologia do painel funil de vendas. É a soma do VrProdutoRateado. Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado)',
  RealAprovado
  COMMENT 'Terminologia do painel funil de vendas, pode ser chamado de "venda aprovadas" ou "real app". É a soma do VrProdutoRateado quando o pedido é aprovado (IdFlagAprovado=true). Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado)',
  VrMargem
  COMMENT 'Margem da venda. É o calculo do VrProdutoRateado retirando os custos de impostos',
  VrReceitaLiquidaSemImpostos
  COMMENT 'Receita liquida retirando os impostos',
  VrDesconto
  COMMENT 'Desconto realizado na venda.')
  WITH METRICS
  LANGUAGE YAML
  COMMENT 'Metricas baseadas no cubo de vendas, com informações de vendas colocadas e aprovadas do site/e-commerce/online/on contendo os valores dos itens vendidos, sendo relatório contendo dia e hora da venda feita'
  AS $$
  version: 0.1

  # --- Example definition with filter, dimensions and measures ---
  #
  source: data_engineering_prd.app_venda.vendacolocadaaprovadasiterateado

  #
  joins: 
    - source: data_engineering_prd.app_venda.mercadoria
      name: merc
      on: merc.CdSkuSite = source.CdSkuSite and merc.StUltimaVersaoMercadoria = 'Y' and not(upper(TRIM(merc.NmSetorGerencial)) in ("BORAVAREJAR", "CARTAO PRESENTE"))
    
    - source: data_engineering_prd.app_venda.canalmarketing 
      name: cm 
      on: source.NmParceiro = cm.DsUtmSource
    
    - source: data_engineering_prd.app_venda.tipoentregasite 
      name: tp_ent
      on: tp_ent.IdTipoEntregaSite = source.IdTipoEntregaSite

    - source: data_engineering_prd.app_venda.meiopagamento 
      name: fma_pgto
      on: fma_pgto.IdMeioPagamento = source.IdMeioPagamento
    
    - source: data_engineering_prd.app_venda.tipocelula 
      name: tp_cel
      on: tp_cel.IdTipoCelula = source.IdTipoCelula
    
    - source: data_engineering_prd.context_site.canalvenda 
      name: canal
      on: source.IdCanalVenda = canal.idcanalvenda and source.NmBandeira = canal.bandeira
    

  filter: source.IdLojista NOT IN (select IdLojista from data_engineering_prd.app_marketplace.cadastrolojista where NmClusterLojista in ('VENDA_DAT', 'Extra/Ztoy', 'Via Varejo', 'DEPÓSITO VV')) and merc.CdSkuSite is not null and tp_ent.IdTipoEntregaSite is not null

  dimensions:
    - name: IdCompra
      expr: source.IdCompra

    - name: IdCompraEntrega
      expr: source.IdCompraEntrega

    - name: CdSkuSite
      expr: merc.CdSkuSite

    - name: CdSkuLoja
      expr: merc.CdSkuLoja
    
    - name: NmSku
      expr: merc.NmSku

    - name: NmMarca
      expr: merc.NmMarca

    - name: DtCompra
      expr: source.DtCompra

    - name: HrVenda
      expr: source.IdHora

    - name: NmSetorGerencial
      expr: trim(merc.NmSetorGerencial)

    - name: NmAgrupamentoDiretoriaSetor
      expr: trim(merc.NmAgrupamentoDiretoriaSetor)

    - name: NmTipoNegocio
      expr: source.NmTipoNegocio

    - name: IdCanalVenda
      expr: source.IdCanalVenda

    - name: DsCanalVenda
      expr: canal.nome

    - name: NmBandeira
      expr: source.NmBandeira

    - name: IdFlagAprovado
      expr: source.IdFlagAprovado

    - name: DtAprovacao
      expr: to_date(source.DtAprovacao, 'yyyyMMdd')

    - name: IdFilial
      expr: source.IdFilial

    - name: IdFilialLoja
      expr: source.IdFilialLoja

    - name: NmUfOrigem
      expr: source.NmUfOrigem

    - name: NmUfEntrega
      expr: source.NmUfEntrega

    - name: NmTipoPagamento
      expr: fma_pgto.NmTipoPagamento

    - name: QtParcela
      expr: fma_pgto.QtParcela

    - name: NmTipoCelula
      expr: tp_cel.NmTipoCelula

    - name: NmTipoEntregaSite
      expr: tp_ent.NmTipoEntregaSite

    - name: DsCanalMarketing
      expr: cm.NmCanalMarketing

  measures:
    # Valor de venda do produto referente a forma de pagamento utilizada. Não inclui os valores de juros e frete.
    - name: VrProdutoRateado
      expr: sum(source.VrProdutoRateado)
    
    - name: VrFreteRateado
      expr: sum(source.VrFreteRateado)
    
    - name: VrJurosRateado
      expr: sum(source.VrJurosRateado)

    # Terminologia do painel funil de vendas, relacionada a vendas captadas sem juros e com frete. É a soma do VrProdutoRateado com VrFreteRateado.
    - name: RealGMV
      expr: sum(coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0))
    
    # Terminologia do painel funil de vendas, relacionada a vendas captadas com juros e frete. É a soma do VrProdutoRateado com VrFreteRateado.
    - name: RealGMVcomJuros
      expr: sum(coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0) + coalesce(source.VrJurosRateado,0))
    
    # Terminologia do painel funil de vendas, relacionada a vendas captadas. É a soma do VrProdutoRateado. Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).
    - name: RealOrders
      expr: sum(case when source.NmTipoNegocio in ('3P','3P_HO') then coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0) else coalesce(source.VrProdutoRateado,0) end)

    # Terminologia do painel funil de vendas, pode ser chamado de "venda aprovadas" ou "real app". É a soma do VrProdutoRateado quando o pedido é aprovado (IdFlagAprovado=true). Nas unidades de negócio 3P e 3P_HO somamos o frete (VrFreteRateado).
    - name: RealAprovado
      expr: sum(case when source.IdFlagAprovado=TRUE and source.NmTipoNegocio in ('3P','3P_HO') then coalesce(source.VrProdutoRateado,0) + coalesce(source.VrFreteRateado,0)
        when source.IdFlagAprovado=TRUE then source.VrProdutoRateado else 0 end)

    # Margem da venda. É o calculo do VrProdutoRateado retirando os custos de impostos.
    - name: VrMargem
      expr: sum(case when merc.CdSetorGerencial<>20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrIcmsVenda +source.VrCustoContabilFilialSkuMarkup) + (source.VrRessarcimentoCreditoPresumido+source.VrRessarcimentoQuebraCadeia+source.VrRessarcimentoSTF) when merc.CdSetorGerencial=20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrCustoContabilFilialSkuMarkup) + (source.VrRessarcimentoCreditoPresumido+source.VrRessarcimentoQuebraCadeia+source.VrRessarcimentoSTF) else 0 end)
    
    # Receita liquida retirando os impostos.
    - name: VrReceitaLiquidaSemImpostos
      expr: sum(case when merc.CdSetorGerencial<>20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS+source.VrIcmsVenda) when merc.CdSetorGerencial=20 and source.NmTipoNegocio not in ('3P','3P_HO') then source.VrProdutoRateado - (source.VrCofins+source.VrPIS) else 0 end)
    
    # Desconto realizado na venda.
    - name: VrDesconto
      expr: sum(source.VrProdutoSemDesconto - source.VrProdutoRateado)

  $$;
 